# Lab 2 — Decision Tree Modeling and Improvement

**Task:** Binary classification of breast tumors as malignant or benign  
**Dataset:** Breast Cancer Wisconsin (Diagnostic)  
**Experiment owner:** Group 9 — Technical Owner (Member A)

This notebook is the project's **single source of truth**. It loads and audits the data, creates one reproducible train/test split, evaluates an untuned baseline tree, tests three improvement methods using cross-validation on the training set, and exports every table and figure used by the report.

## Experiment map

1. Setup and dataset audit
2. Experimental protocol
3. Baseline Decision Tree
4. Tree structure and decision-rule analysis
5. Improvement 1 — `max_depth`
6. Improvement 2 — `min_samples_leaf`
7. Improvement 3 — cost-complexity pruning
8. Fair comparison, winner, and exported handoff files


## 1. Setup and reproducibility

All models use the same `RANDOM_STATE` and the same stratified 80/20 split. Hyperparameters are selected only through five-fold cross-validation on the training partition. The test partition is reserved for the final comparison.

The dataset is bundled with scikit-learn, so no machine-specific or absolute data path is required.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

from sklearn.datasets import load_breast_cancer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

RANDOM_STATE = 42
TEST_SIZE = 0.20
CV_FOLDS = 5

RESULTS_DIR = Path("results")
FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 4)
plt.style.use("seaborn-v0_8-whitegrid")

print(f"Artifacts will be written to: {RESULTS_DIR.resolve()}")


## 2. Dataset selection and description

The [Breast Cancer Wisconsin (Diagnostic) dataset](https://doi.org/10.24432/C5DW2B) contains numeric characteristics computed from digitized images of fine-needle aspirates of breast masses. The target contains two classes: **malignant (0)** and **benign (1)**.

It is appropriate for Decision Tree modeling because it is a supervised binary-classification problem, its numeric measurements support threshold-based splits, and the resulting rules can be inspected directly. This implementation checks missing values and duplicates. Scaling is unnecessary because a Decision Tree compares one feature with a threshold at each split.


In [ ]:
data = load_breast_cancer(as_frame=True)
X = data.data.copy()
y = data.target.copy()

dataset_summary = pd.DataFrame(
    {
        "Item": [
            "Samples",
            "Input features",
            "Target",
            "Classes",
            "Missing values",
            "Duplicate feature rows",
        ],
        "Value": [
            len(X),
            X.shape[1],
            "diagnosis",
            "malignant (0), benign (1)",
            int(X.isna().sum().sum()),
            int(X.duplicated().sum()),
        ],
    }
)

class_distribution = (
    y.value_counts()
    .rename(index={0: "malignant", 1: "benign"})
    .rename("Samples")
    .to_frame()
)
class_distribution["Percent"] = 100 * class_distribution["Samples"] / len(y)

display(dataset_summary)
display(class_distribution)
display(X.head())


### Data-quality decision

There are no missing values, so imputation is not required. All 30 predictors are numeric and no categorical encoding is needed. The original features are retained so the learned thresholds remain interpretable. The moderate class imbalance is handled by a stratified split and by reporting class-specific malignant recall in addition to weighted metrics.


## 3. Experimental protocol

- **Training set:** 80% of the observations
- **Test set:** 20% of the observations
- **Stratification:** preserves the malignant/benign ratio
- **Hyperparameter selection:** 5-fold cross-validation on training data only
- **Primary metric:** test accuracy and its complement, error rate
- **Supporting metrics:** weighted precision/recall/F1 and malignant-class recall

This design uses exactly one held-out test set for all models and avoids selecting hyperparameters directly from test performance.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

split_summary = pd.DataFrame(
    {
        "Partition": ["Training", "Testing"],
        "Samples": [len(X_train), len(X_test)],
        "Malignant": [(y_train == 0).sum(), (y_test == 0).sum()],
        "Benign": [(y_train == 1).sum(), (y_test == 1).sum()],
    }
)
display(split_summary)


In [ ]:
def evaluate_model(name, model, key_parameter, cv_accuracy=np.nan):
    # Return one consistent row of performance and complexity metrics.
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    test_accuracy = accuracy_score(y_test, test_pred)

    return {
        "Model": name,
        "Key Parameter": key_parameter,
        "CV Accuracy": cv_accuracy,
        "Train Accuracy": accuracy_score(y_train, train_pred),
        "Test Accuracy": test_accuracy,
        "Error Rate": 1 - test_accuracy,
        "Weighted Precision": precision_score(
            y_test, test_pred, average="weighted", zero_division=0
        ),
        "Weighted Recall": recall_score(
            y_test, test_pred, average="weighted", zero_division=0
        ),
        "Weighted F1": f1_score(
            y_test, test_pred, average="weighted", zero_division=0
        ),
        "Malignant Precision": precision_score(
            y_test, test_pred, pos_label=0, zero_division=0
        ),
        "Malignant Recall": recall_score(
            y_test, test_pred, pos_label=0, zero_division=0
        ),
        "Malignant F1": f1_score(
            y_test, test_pred, pos_label=0, zero_division=0
        ),
        "Tree Depth": model.get_depth(),
        "Leaves": model.get_n_leaves(),
        "Nodes": model.tree_.node_count,
    }


def save_tree(model, filename, title, *, max_depth=None, figsize=(24, 12), fontsize=8):
    # Save a report-ready tree figure and show it in the notebook.
    fig, ax = plt.subplots(figsize=figsize)
    plot_tree(
        model,
        feature_names=X.columns,
        class_names=data.target_names,
        filled=True,
        rounded=True,
        max_depth=max_depth,
        fontsize=fontsize,
        ax=ax,
    )
    ax.set_title(title, fontsize=16, pad=14)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / filename, dpi=220, bbox_inches="tight")
    plt.show()
    plt.close(fig)


## 4. Baseline Decision Tree

The baseline uses scikit-learn's default `DecisionTreeClassifier` settings with only the fixed random seed supplied. It therefore provides an honest, untuned reference rather than an intentionally weakened model.


In [ ]:
baseline = DecisionTreeClassifier(random_state=RANDOM_STATE)
baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)
baseline_row = evaluate_model("Baseline", baseline, "default")

display(pd.DataFrame([baseline_row]).round(4))
display(
    pd.DataFrame(
        classification_report(
            y_test,
            baseline_pred,
            target_names=data.target_names,
            output_dict=True,
            zero_division=0,
        )
    ).T.round(4)
)


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    baseline_pred,
    display_labels=data.target_names,
    cmap="Blues",
    colorbar=False,
    ax=ax,
)
ax.set_title("Baseline Decision Tree — Confusion Matrix")
fig.tight_layout()
fig.savefig(
    FIGURES_DIR / "baseline_confusion_matrix.png",
    dpi=220,
    bbox_inches="tight",
)
plt.show()
plt.close(fig)


In [ ]:
# The full tree is exported for the report appendix/handoff package.
save_tree(
    baseline,
    "baseline_tree_full.png",
    "Baseline Decision Tree — Full Structure",
    figsize=(32, 18),
    fontsize=7,
)


In [ ]:
# The first three levels are easier to read and discuss in the main report.
save_tree(
    baseline,
    "baseline_tree_readable.png",
    "Baseline Decision Tree — First Three Levels",
    max_depth=3,
    figsize=(24, 12),
    fontsize=9,
)


## 5. Analysis of the baseline tree

The following cells expose both global feature importance and the first decision rules. These outputs support concrete interpretation rather than relying only on the final accuracy.


In [ ]:
baseline_structure = pd.DataFrame(
    {
        "Measure": ["Training accuracy", "Test accuracy", "Train–test gap", "Depth", "Leaves", "Nodes"],
        "Value": [
            baseline_row["Train Accuracy"],
            baseline_row["Test Accuracy"],
            baseline_row["Train Accuracy"] - baseline_row["Test Accuracy"],
            baseline.get_depth(),
            baseline.get_n_leaves(),
            baseline.tree_.node_count,
        ],
    }
)
display(baseline_structure.round(4))

print("First decision rules (top three levels):")
print(export_text(baseline, feature_names=list(X.columns), max_depth=2))


In [ ]:
feature_importance = pd.Series(
    baseline.feature_importances_, index=X.columns, name="Importance"
).sort_values(ascending=False)

display(feature_importance.head(10).to_frame())

fig, ax = plt.subplots(figsize=(9, 6))
feature_importance.head(10).sort_values().plot(kind="barh", ax=ax, color="#3976af")
ax.set_xlabel("Feature importance")
ax.set_ylabel("")
ax.set_title("Top 10 Features — Baseline Decision Tree")
fig.tight_layout()
fig.savefig(
    FIGURES_DIR / "baseline_feature_importance.png",
    dpi=220,
    bbox_inches="tight",
)
plt.show()
plt.close(fig)


### Baseline interpretation

- The root split is **`worst radius <= 16.80`**, and `worst radius` accounts for most of the tree's impurity reduction. This indicates that the size of the most extreme observed nuclei is the dominant first separator for this training split.
- Subsequent high-level decisions use `worst concave points`, `area error`, `worst texture`, and `texture error`, showing that shape irregularity and texture complement the size measurement.
- The baseline reaches **100% training accuracy** but only about **91.23% test accuracy**. Together with depth 7 and 19 leaves, the 8.77 percentage-point gap is evidence that the unconstrained tree fits training-specific detail.
- The confusion matrix must be read by class: the baseline misses 3 of 42 malignant cases and incorrectly flags 7 of 72 benign cases. Accuracy alone would hide this error distribution.


## 6. Improvement 1 — limit `max_depth`

Limiting depth prevents the tree from continuing to form training-specific branches. Candidate depths are selected by five-fold cross-validation on the training data. A depth that is too small can underfit; a depth that is too large approaches the baseline.


In [ ]:
depth_values = [2, 3, 4, 5, 6, 7, 8, 10]
depth_search = GridSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    param_grid={"max_depth": depth_values},
    cv=CV_FOLDS,
    scoring="accuracy",
    n_jobs=1,
    return_train_score=True,
)
depth_search.fit(X_train, y_train)
depth_model = depth_search.best_estimator_

depth_cv = pd.DataFrame(depth_search.cv_results_)[
    ["param_max_depth", "mean_train_score", "mean_test_score", "std_test_score"]
].rename(
    columns={
        "param_max_depth": "max_depth",
        "mean_train_score": "Mean Train Accuracy",
        "mean_test_score": "Mean CV Accuracy",
        "std_test_score": "CV Std. Dev.",
    }
)
display(depth_cv.round(4))
print("Selected parameters:", depth_search.best_params_)

#update:
# Đánh giá mô hình trên tập Test (Accuracy, Error Rate, F1, Complexity)
depth_row = evaluate_model(
    name="Max Depth", 
    model=depth_model, 
    key_parameter=f"max_depth={depth_search.best_params_['max_depth']}", 
    cv_accuracy=depth_search.best_score_
)

# print result
display(pd.DataFrame([depth_row]).round(4))

# print tree
save_tree(
    model=depth_model, 
    filename="tree_improvement_1.png", 
    title=f"Improvement 1: max_depth={depth_search.best_params_['max_depth']}"
)

## 7. Improvement 2 — increase `min_samples_leaf`

This regularization requires each terminal node to contain enough training observations. It discourages leaves that memorize only one or two cases while preserving the tree's ability to grow when supported by data.


In [ ]:
leaf_values = [1, 2, 3, 4, 5, 6, 8, 10, 15]
leaf_search = GridSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    param_grid={"min_samples_leaf": leaf_values},
    cv=CV_FOLDS,
    scoring="accuracy",
    n_jobs=1,
    return_train_score=True,
)
leaf_search.fit(X_train, y_train)
leaf_model = leaf_search.best_estimator_

leaf_cv = pd.DataFrame(leaf_search.cv_results_)[
    ["param_min_samples_leaf", "mean_train_score", "mean_test_score", "std_test_score"]
].rename(
    columns={
        "param_min_samples_leaf": "min_samples_leaf",
        "mean_train_score": "Mean Train Accuracy",
        "mean_test_score": "Mean CV Accuracy",
        "std_test_score": "CV Std. Dev.",
    }
)

display(leaf_cv.round(4))
print("Selected parameters:", leaf_search.best_params_)

#update:
#chuẩn hóa metric
leaf_row = evaluate_model(
    name="Min Samples Leaf", 
    model=leaf_model, 
    key_parameter=f"min_samples_leaf={leaf_search.best_params_['min_samples_leaf']}", 
    cv_accuracy=leaf_search.best_score_
)

display(pd.DataFrame([leaf_row]).round(4))

save_tree(
    model=leaf_model, 
    filename="tree_improvement_2.png", 
    title=f"Improvement 2: min_samples_leaf={leaf_search.best_params_['min_samples_leaf']}"
)


## 8. Improvement 3 — cost-complexity pruning

Cost-complexity pruning penalizes additional branches through `ccp_alpha`. Candidate alpha values come from the pruning path of the training data, and cross-validation selects the trade-off between predictive fit and structural simplicity.


In [ ]:
pruning_base = DecisionTreeClassifier(random_state=RANDOM_STATE)
pruning_path = pruning_base.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = np.unique(pruning_path.ccp_alphas[:-1])

pruning_search = GridSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    param_grid={"ccp_alpha": ccp_alphas},
    cv=CV_FOLDS,
    scoring="accuracy",
    n_jobs=1,
    return_train_score=True,
)
pruning_search.fit(X_train, y_train)
pruned_model = pruning_search.best_estimator_

pruning_cv = pd.DataFrame(pruning_search.cv_results_)[
    ["param_ccp_alpha", "mean_train_score", "mean_test_score", "std_test_score"]
].rename(
    columns={
        "param_ccp_alpha": "ccp_alpha",
        "mean_train_score": "Mean Train Accuracy",
        "mean_test_score": "Mean CV Accuracy",
        "std_test_score": "CV Std. Dev.",
    }
)
display(pruning_cv.round(5))
print("Selected parameters:", pruning_search.best_params_)

#update:

pruning_row = evaluate_model(
    name="Pruning", 
    model=pruned_model, 
    key_parameter=f"ccp_alpha={pruning_search.best_params_['ccp_alpha']:.5f}", 
    cv_accuracy=pruning_search.best_score_
)

display(pd.DataFrame([pruning_row]).round(4))

save_tree(
    model=pruned_model, 
    filename="tree_improvement_3.png", 
    title=f"Improvement 3: ccp_alpha={pruning_search.best_params_['ccp_alpha']:.5f}"
)


In [ ]:
# Cross-validation curves make the selection process visible.
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

axes[0].errorbar(
    depth_cv["max_depth"],
    depth_cv["Mean CV Accuracy"],
    yerr=depth_cv["CV Std. Dev."],
    marker="o",
    capsize=3,
)
axes[0].set(title="Tune max_depth", xlabel="max_depth", ylabel="CV accuracy")

axes[1].errorbar(
    leaf_cv["min_samples_leaf"],
    leaf_cv["Mean CV Accuracy"],
    yerr=leaf_cv["CV Std. Dev."],
    marker="o",
    capsize=3,
    color="#d26b34",
)
axes[1].set(title="Tune min_samples_leaf", xlabel="min_samples_leaf", ylabel="CV accuracy")

axes[2].errorbar(
    pruning_cv["ccp_alpha"],
    pruning_cv["Mean CV Accuracy"],
    yerr=pruning_cv["CV Std. Dev."],
    marker="o",
    capsize=3,
    color="#3a9668",
)
axes[2].set(title="Tune cost-complexity pruning", xlabel="ccp_alpha", ylabel="CV accuracy")

for ax in axes:
    ax.set_ylim(0.86, 0.97)

fig.suptitle("Five-Fold Cross-Validation on the Training Set", fontsize=15)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "hyperparameter_tuning.png", dpi=220, bbox_inches="tight")
plt.show()
plt.close(fig)


## 9. Final comparison and model selection

All four fitted models are evaluated on the same held-out test set. The selection rule is explicit:

1. Maximize test accuracy.
2. If accuracy is tied, prefer malignant-class recall because a missed malignant case is the more consequential error in this task.
3. Then compare weighted F1 and structural simplicity.

The test set is used here for the final model comparison, not to tune the internal hyperparameters of any individual method.


In [ ]:
model_specs = [
    ("Baseline", baseline, "default", np.nan),
    (
        "Max Depth",
        depth_model,
        f"max_depth={depth_search.best_params_['max_depth']}",
        depth_search.best_score_,
    ),
    (
        "Min Samples Leaf",
        leaf_model,
        f"min_samples_leaf={leaf_search.best_params_['min_samples_leaf']}",
        leaf_search.best_score_,
    ),
    (
        "Pruning",
        pruned_model,
        f"ccp_alpha={pruning_search.best_params_['ccp_alpha']:.6f}",
        pruning_search.best_score_,
    ),
]

results_df = pd.DataFrame(
    [evaluate_model(name, model, parameter, cv_score) for name, model, parameter, cv_score in model_specs]
)

max_accuracy = results_df["Test Accuracy"].max()
accuracy_tied = results_df[np.isclose(results_df["Test Accuracy"], max_accuracy)]
winner_row = accuracy_tied.sort_values(
    by=["Malignant Recall", "Weighted F1", "Leaves", "Tree Depth"],
    ascending=[False, False, True, True],
).iloc[0]
best_model_name = winner_row["Model"]
best_model = dict((name, model) for name, model, _, _ in model_specs)[best_model_name]

display(results_df.round(4))
display(Markdown(
    f"**Selected model: {best_model_name}.** "
    f"Test accuracy = {winner_row['Test Accuracy']:.2%}, "
    f"error rate = {winner_row['Error Rate']:.2%}, "
    f"malignant recall = {winner_row['Malignant Recall']:.2%}."
))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
colors = ["#808080", "#3976af", "#d26b34", "#3a9668"]

results_df.plot(
    x="Model",
    y=["Train Accuracy", "Test Accuracy"],
    kind="bar",
    ylim=(0.85, 1.01),
    ax=axes[0],
    color=["#b8c8d8", "#3976af"],
)
axes[0].set_title("Predictive Performance")
axes[0].set_ylabel("Accuracy")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=20)

axes[1].bar(results_df["Model"], results_df["Leaves"], color=colors)
axes[1].set_title("Model Complexity")
axes[1].set_ylabel("Number of leaves")
axes[1].tick_params(axis="x", rotation=20)

fig.suptitle("Baseline and Improved Decision Trees", fontsize=15)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "model_comparison.png", dpi=220, bbox_inches="tight")
plt.show()
plt.close(fig)


In [ ]:
# Export all tree variants. The Max Depth tree is the selected final model.
save_tree(
    depth_model,
    "tree_max_depth.png",
    "Improved Decision Tree — max_depth",
    figsize=(26, 14),
    fontsize=8,
)
save_tree(
    leaf_model,
    "tree_min_samples_leaf.png",
    "Improved Decision Tree — min_samples_leaf",
    figsize=(30, 16),
    fontsize=7,
)
save_tree(
    pruned_model,
    "tree_pruned.png",
    "Improved Decision Tree — Cost-Complexity Pruning",
    figsize=(22, 12),
    fontsize=9,
)


### Interpretation of the three improvements

- **`max_depth=4`:** test accuracy rises from 91.23% to 93.86%, error falls from 8.77% to 6.14%, and the tree shrinks from 19 to 11 leaves. It preserves malignant recall at 92.86% while removing deeper training-specific splits.
- **`min_samples_leaf=3`:** the tree becomes smaller, but test accuracy remains 91.23% and malignant recall falls to 90.48%. The constraint regularizes the model without improving generalization on this split.
- **Pruning (`ccp_alpha≈0.005934`):** test accuracy also reaches 93.86% with only 6 leaves and depth 3. It is the simplest competitive model, but it detects one fewer malignant test case than the max-depth model.
- **Winner:** Max Depth is selected because it ties Pruning on accuracy while obtaining higher malignant recall (92.86% versus 90.48%) and a slightly higher weighted F1 score. If interpretability or deployment size were the dominant objective, Pruning would be a reasonable alternative.


## 10. Export the Technical Owner handoff package

The final cell writes the comparison CSV, selected-parameter table, and a concise `SUMMARY.md`. All report figures have already been saved under `results/figures/` with stable names.


In [ ]:
#update:
results_df = pd.DataFrame([baseline_row, depth_row, leaf_row, pruning_row]) #gom kết quả
max_accuracy = results_df['Test Accuracy'].max()
best_model_name = "Max Depth"
#===

results_df.to_csv(RESULTS_DIR / "model_comparison.csv", index=False)

hyperparameter_summary = pd.DataFrame(
    [
        {
            "Method": "Max Depth",
            "Selected Parameter": f"max_depth={depth_search.best_params_['max_depth']}",
            "Best CV Accuracy": depth_search.best_score_,
        },
        {
            "Method": "Min Samples Leaf",
            "Selected Parameter": f"min_samples_leaf={leaf_search.best_params_['min_samples_leaf']}",
            "Best CV Accuracy": leaf_search.best_score_,
        },
        {
            "Method": "Pruning",
            "Selected Parameter": f"ccp_alpha={pruning_search.best_params_['ccp_alpha']:.6f}",
            "Best CV Accuracy": pruning_search.best_score_,
        },
    ]
)
hyperparameter_summary.to_csv(RESULTS_DIR / "hyperparameter_summary.csv", index=False)

summary = f'''# Lab 2 Decision Tree — Technical Handoff

## Reproducibility

- Dataset: Breast Cancer Wisconsin (Diagnostic), loaded from scikit-learn
- Source: https://doi.org/10.24432/C5DW2B
- Samples/features: {len(X)} / {X.shape[1]}
- Classes: malignant (0), benign (1)
- Split: {int((1-TEST_SIZE)*100)}/{int(TEST_SIZE*100)}, stratified
- Random state: {RANDOM_STATE}
- Hyperparameter selection: {CV_FOLDS}-fold cross-validation on training data

## Locked results

- Baseline: test accuracy {baseline_row['Test Accuracy']:.4f}, error rate {baseline_row['Error Rate']:.4f}, depth {baseline.get_depth()}, leaves {baseline.get_n_leaves()}
- Max Depth: max_depth={depth_search.best_params_['max_depth']}, test accuracy {results_df.loc[results_df['Model'].eq('Max Depth'), 'Test Accuracy'].iloc[0]:.4f}
- Min Samples Leaf: min_samples_leaf={leaf_search.best_params_['min_samples_leaf']}, test accuracy {results_df.loc[results_df['Model'].eq('Min Samples Leaf'), 'Test Accuracy'].iloc[0]:.4f}
- Pruning: ccp_alpha={pruning_search.best_params_['ccp_alpha']:.6f}, test accuracy {results_df.loc[results_df['Model'].eq('Pruning'), 'Test Accuracy'].iloc[0]:.4f}
- Selected model: {best_model_name}

## Selection rationale

Max Depth and Pruning tie at {max_accuracy:.2%} test accuracy. Max Depth is selected because malignant recall is {results_df.loc[results_df['Model'].eq('Max Depth'), 'Malignant Recall'].iloc[0]:.2%}, compared with {results_df.loc[results_df['Model'].eq('Pruning'), 'Malignant Recall'].iloc[0]:.2%} for Pruning, and its weighted F1 is slightly higher. Pruning remains the simplest competitive alternative.

## Files

- `Decision_Tree_2.ipynb`: executed source notebook and embedded outputs
- `results/model_comparison.csv`: authoritative model metrics
- `results/hyperparameter_summary.csv`: selected settings and CV scores
- `results/figures/`: report-ready trees, confusion matrix, feature importance, tuning, and comparison figures

All report and presentation numbers should be copied from these files.
'''
(RESULTS_DIR / "SUMMARY.md").write_text(summary, encoding="utf-8")

print("Technical handoff exported successfully:")
for path in sorted(RESULTS_DIR.rglob("*")):
    if path.is_file():
        print(" -", path.as_posix())


## 11. Technical conclusion

The unrestricted baseline shows a clear train–test gap. Cross-validated structural regularization improves generalization: both limiting depth and pruning reduce the test error from 8.77% to 6.14%. The selected `max_depth=4` tree offers the best balance for this experiment because it retains the higher malignant recall, while the pruned tree offers the most compact representation.

**Technical Owner status:** notebook logic, reproducible execution, metrics, figures, comparison table, winner rationale, and handoff summary are complete. The report and video should reuse these locked outputs without changing the split, seed, parameters, or model names.
